In [1]:
##Part 1 — Install Libraries
!pip install -q google-genai pydantic pandas sentence-transformers faiss-cpu tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 64.7 MB/s eta 0:00:00


In [2]:

#Part 2 — Import Libraries
import json
import pandas as pd
from tqdm import tqdm

from google import genai
from pydantic import BaseModel, ValidationError, ConfigDict
from enum import Enum

In [22]:
!pip install -q python-dotenv

In [69]:
##Part 3 — Enter Gemini API Key

from google.colab import userdata
userdata.get('GEMINI_API_KEY')

'AQ.Ab8RN6KfipEv5NsiEDgU1e4QQgF8kEnkP2l_PPrlA2B_HAO4Aw'

In [68]:
##Part 4 — Initialize Gemini Client
from google import genai

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL = "models/gemini-flash-latest"

response = client.models.generate_content(
    model=MODEL,
    contents="Say Hello"
)

print(response.text)

Hello! How can I help you today?


In [11]:
!pip show google-genai

Name: google-genai
Version: 2.11.0
Summary: GenAI Python SDK
Home-page: https://github.com/googleapis/python-genai
Author: 
Author-email: Google LLC <googleapis-packages@google.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: anyio, distro, google-auth, httpx, pydantic, requests, sniffio, tenacity, typing-extensions, websockets
Required-by: google-adk


In [27]:
##Part 5 — Test API
response = client.models.generate_content(
    model=MODEL,
    contents="Say Hello"
)

print(response.text)

Hello! How can I help you today?


In [28]:
##Part 6 — Create Dataset (15 Records)
records = [

"Order #4512 has not arrived even after ten days. Please help urgently.",

"The shoes look amazing but the size is too small. I want an exchange.",

"My payment was deducted twice but I received only one order confirmation.",

"I love the product quality. Delivery was also very fast.",

"The package arrived damaged and the item is broken.",

"I accidentally entered the wrong shipping address. Please update it.",

"Customer support solved my issue quickly. Great experience.",

"The fabric quality is poor and I need a refund immediately.",

"I have not received my refund for the returned product.",

"Can I cancel my order before it gets shipped?",

"The delivery executive was very polite and helpful.",

"The color is completely different from the website images.",

"My coupon code is not working during checkout.",

"I received someone else's order instead of mine.",

"Very satisfied with the purchase. Will buy again."

]

In [29]:
##Part 7 — Save Dataset
df = pd.DataFrame({"text": records})

df.to_csv("customer_messages.csv", index=False)

df.head()

,text
0,Order #4512 has not arrived even after ten day...
1,The shoes look amazing but the size is too sma...
2,My payment was deducted twice but I received o...
3,I love the product quality. Delivery was also ...
4,The package arrived damaged and the item is br...


In [30]:
##Part 8 — Define Allowed Values
class Category(str, Enum):

    Delivery = "Delivery"

    Payment = "Payment"

    Refund = "Refund"

    Exchange = "Exchange"

    Complaint = "Complaint"

    Praise = "Praise"

    Cancellation = "Cancellation"

    Other = "Other"


class Urgency(str, Enum):

    Low = "Low"

    Medium = "Medium"

    High = "High"

In [31]:
##Part 9 — Create Pydantic Schema
class TicketSchema(BaseModel):

    model_config = ConfigDict(extra="forbid")

    category: Category

    urgency: Urgency

    sentiment: str

    summary: str

In [32]:

##Part 10 — Prompt Template
PROMPT = """
You are a customer support analyst.

Extract structured information.

Return ONLY valid JSON.

Required fields:

category

urgency

sentiment

summary

Allowed Categories:

Delivery

Payment

Refund

Exchange

Complaint

Praise

Cancellation

Other

Allowed Urgency:

Low

Medium

High

Customer message:

{text}
"""

In [33]:
##Part 11 — LLM Function
def extract_information(text):

    prompt = PROMPT.format(text=text)

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return response.text

In [34]:

##Part 12 — JSON Parser
def parse_json(response):

    response = response.replace("```json","")

    response = response.replace("```","")

    response = response.strip()

    return json.loads(response)

In [35]:
#Part 2 – Process All Records & Validate
##Step 13 – Process All Records
validated_results = []
failed_records = []

for i, text in enumerate(records):

    print(f"\nProcessing Record {i+1}")

    try:

        response = extract_information(text)

        data = parse_json(response)

        ticket = TicketSchema(**data)

        validated_results.append(ticket.model_dump())

        print("✓ Valid")

    except Exception as e:

        failed_records.append({
            "record": text,
            "error": str(e)
        })

        print("✗ Failed")
        print(e)


Processing Record 1
✓ Valid

Processing Record 2
✓ Valid

Processing Record 3
✓ Valid

Processing Record 4
✓ Valid

Processing Record 5
✓ Valid

Processing Record 6
✓ Valid

Processing Record 7
✓ Valid

Processing Record 8
✓ Valid

Processing Record 9
✓ Valid

Processing Record 10
✓ Valid

Processing Record 11
✓ Valid

Processing Record 12
✓ Valid

Processing Record 13
✗ Failed
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 17.011187946s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API 

In [36]:
##Step 14 – Display Valid Results
validated_df = pd.DataFrame(validated_results)

validated_df

,category,urgency,sentiment,summary
0,Category.Delivery,Urgency.High,Negative,Order #4512 has not arrived after ten days and...
1,Category.Exchange,Urgency.Medium,Mixed,Customer likes the shoes but needs an exchange...
2,Category.Payment,Urgency.High,Negative,Customer was charged twice for a single order ...
3,Category.Praise,Urgency.Low,Positive,Customer expressed satisfaction with the produ...
4,Category.Delivery,Urgency.High,Negative,The customer received a damaged package contai...
5,Category.Delivery,Urgency.High,Neutral,Customer accidentally entered the wrong shippi...
6,Category.Praise,Urgency.Low,Positive,Customer praised customer support for resolvin...
7,Category.Refund,Urgency.High,Negative,Customer is dissatisfied with the fabric quali...
8,Category.Refund,Urgency.Medium,Negative,Customer has not received the refund for their...
9,Category.Cancellation,Urgency.Medium,Neutral,Customer is asking if an order can be cancelle...


In [37]:

##Step 15 – Save Results
validated_df.to_csv("validated_results.csv", index=False)

with open("validated_results.json","w") as f:
    json.dump(validated_results,f,indent=4)

print("Files saved successfully.")
#Required Validation Failure (Assignment Requirement)

Files saved successfully.


In [38]:
##Step 16 – Hardcoded Invalid Fixture
bad_output = {

    "category":"delivery",

    "urgency":"Very High",

    "sentiment":"Negative",

    "summary":"Package delayed",

    "extra_field":"Not Allowed"

}

In [39]:
##Step 17 – Validate Fixture
try:

    TicketSchema(**bad_output)

except ValidationError as e:

    print("Validation Failed")

    print(e)
#Expected output:
#Validation Failed



Validation Failed
3 validation errors for TicketSchema
category
  Input should be 'Delivery', 'Payment', 'Refund', 'Exchange', 'Complaint', 'Praise', 'Cancellation' or 'Other' [type=enum, input_value='delivery', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum
urgency
  Input should be 'Low', 'Medium' or 'High' [type=enum, input_value='Very High', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum
extra_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='Not Allowed', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden


In [40]:
##Step 18 – Normalize & Retry
category_map = {
    "delivery":"Delivery",
    "payment":"Payment",
    "refund":"Refund",
    "exchange":"Exchange",
    "complaint":"Complaint",
    "praise":"Praise",
    "cancellation":"Cancellation",
    "other":"Other"
}

urgency_map = {
    "very high":"High",
    "high":"High",
    "medium":"Medium",
    "low":"Low"
}

fixed = bad_output.copy()

fixed["category"] = category_map.get(
    fixed["category"].lower(),
    "Other"
)

fixed["urgency"] = urgency_map.get(
    fixed["urgency"].lower(),
    "High"
)

fixed.pop("extra_field",None)

ticket = TicketSchema(**fixed)

print(ticket)

category=<Category.Delivery: 'Delivery'> urgency=<Urgency.High: 'High'> sentiment='Negative' summary='Package delayed'


In [41]:
##PART 3 — RAG Pipeline
#Step 19 – Install Packages
!pip install -q sentence-transformers faiss-cpu

In [42]:
##Step 20 – Imports
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [43]:
##Step 21 – Create Documents
#Create 5 text files (doc1.txt to doc5.txt) with a total of 2000+ words.

import os

# Create the folder
os.makedirs("documents", exist_ok=True)

# Document contents

documents = [

"""
Customer Support Policy

Refunds are processed within seven business days after the returned item passes inspection.

Customers may request exchanges within ten days.

Damaged products qualify for free replacement.

""" * 25,

"""
Shipping Policy

Standard delivery takes 5–7 business days.

Express delivery takes 2 days.

Customers receive tracking numbers after dispatch.

""" * 25,

"""
Payment Policy

UPI

Credit Card

Debit Card

Wallet

COD

Double payment issues are refunded automatically.

""" * 25,

"""
Returns Policy

Returned products should be unused.

Original packaging is required.

Refunds begin after warehouse verification.

""" * 25,

"""
Customer Service

Support hours are Monday to Saturday.

Email replies within 24 hours.

Phone support available.

""" * 25

]

# Save each document as a text file
for i, content in enumerate(documents, start=1):
    file_path = f"documents/doc{i}.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ 5 documents created successfully!")

# Verify files
print("\nFiles in 'documents' folder:")
print(os.listdir("documents"))

✅ 5 documents created successfully!

Files in 'documents' folder:
['doc2.txt', 'doc5.txt', 'doc4.txt', 'doc1.txt', 'doc3.txt']


In [47]:
#from google.colab import drive
#drive.mount('/content/drive')

ValueError: mount failed

In [48]:
##Step 22 – Chunk Function
def chunk_documents(documents, chunk_size=150):

    chunks=[]

    for doc in documents:

        words=doc.split()

        for i in range(0,len(words),chunk_size):

            chunks.append(" ".join(words[i:i+chunk_size]))

    return chunks

In [49]:
##Step 23 – Create Chunks
chunks = chunk_documents(documents)

print(len(chunks))

18


In [50]:
##Step 24 – Load Embedding Model
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [51]:
##Step 25 – Generate Embeddings
embeddings = embedding_model.encode(chunks)

embeddings = np.array(embeddings).astype("float32")

In [52]:
##Step 26 – Create FAISS Index
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print(index.ntotal)

18


In [53]:
###Part 3 — RAG Retrieval & Grounded Answer Generation
##Step 27 — Retrieval Function
def retrieve(query, k=3):

    query_embedding = embedding_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, k)

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return retrieved_chunks

In [54]:
##Step 28 — Gemini Grounded Answer Function
def rag_answer(query):

    retrieved = retrieve(query)

    context = "\n\n".join(retrieved)

    prompt = f"""
You are an AI assistant.

Answer ONLY using the information provided below.

If the answer is not present, reply:

"I could not find this information in the provided documents."

Context:

{context}

Question:

{query}

Answer:
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt
    )

    return retrieved, response.text

In [55]:
##Step 29 — Test Query
query = "How many days does a refund take?"

chunks_found, answer = rag_answer(query)

print("Retrieved Chunks:\n")

for i, chunk in enumerate(chunks_found,1):
    print("="*80)
    print(f"Chunk {i}")
    print(chunk)

print("\n")
print("="*80)
print("Generated Answer:\n")
print(answer)

Retrieved Chunks:

Chunk 1
within ten days. Damaged products qualify for free replacement. Customer Support Policy Refunds are processed within seven business days after the returned item passes inspection. Customers may request exchanges within ten days. Damaged products qualify for free replacement. Customer Support Policy Refunds are processed within seven business days after the returned item passes inspection. Customers may request exchanges within ten days. Damaged products qualify for free replacement. Customer Support Policy Refunds are processed within seven business days after the returned item passes inspection. Customers may request exchanges within ten days. Damaged products qualify for free replacement. Customer Support Policy Refunds are processed within seven business days after the returned item passes inspection. Customers may request exchanges within ten days. Damaged products qualify for free replacement.
Chunk 2
processed within seven business days after the return

In [56]:
##Step 30 — Five Example Queries
queries = [

    "How many days does a refund take?",

    "What payment methods are available?",

    "Can I exchange a damaged product?",

    "When will I receive my tracking number?",

    "When is customer support available?"

]

In [58]:

##Step 31 — Run All Queries
demo=[]

for q in queries:

    retrieved, ans = rag_answer(q)

    demo.append({

        "Question":q,

        "Retrieved Chunks":"\n\n".join(retrieved),

        "Answer":ans

    })

    print("="*100)

    print("QUESTION")

    print(q)

    print("\n")

    print("ANSWER")

    print(ans)

    print("\n")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-3.6-flash\nPlease retry in 35.323109883s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-flash'

In [59]:
##Step 32 — Save Demo Output
demo_df = pd.DataFrame(demo)

demo_df.to_csv("rag_demo_output.csv",index=False)

demo_df

""


In [60]:
##Step 33 — Save JSON
with open("rag_demo_output.json","w") as f:

    json.dump(demo,f,indent=4)

print("Saved Successfully")

Saved Successfully


In [61]:
##Step 34 — Display Retrieved Chunks for README
for item in demo:

    print("="*100)

    print("QUESTION:")

    print(item["Question"])

    print()

    print("RETRIEVED CHUNKS:")

    print(item["Retrieved Chunks"][:700])

    print()

    print("ANSWER:")

    print(item["Answer"])

In [62]:
##Step 35 — Show Generated Files
import os

for file in os.listdir():

    if file.endswith(".csv") or file.endswith(".json"):
        print(file)

rag_demo_output.csv
validated_results.csv
rag_demo_output.json
validated_results.json
customer_messages.csv


In [63]:
##Step 36 — Download Files (Google Colab)
from google.colab import files

files.download("validated_results.csv")
files.download("validated_results.json")
files.download("rag_demo_output.csv")
files.download("rag_demo_output.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [64]:
%%writefile requirements.txt
google-genai>=1.30.0
pydantic>=2.7.0
sentence-transformers>=3.0.0
faiss-cpu>=1.8.0
pandas>=2.2.0
numpy>=1.26.0
scikit-learn>=1.5.0
python-dotenv>=1.0.1
tqdm>=4.66.0

Writing requirements.txt
